In [1]:
library(dplyr, warn.conflicts = FALSE)
library(tidyr, warn.conflicts = FALSE)
library(purrr, warn.conflicts = FALSE)
library(tibble, warn.conflicts = FALSE)
#library(furrr)
library(future)
library(readr)
library(edgeR)
library(limma)
library(dplyr)
library(statmod)
library(data.table)

Loading required package: limma


Attaching package: ‘data.table’


The following object is masked from ‘package:purrr’:

    transpose


The following objects are masked from ‘package:dplyr’:

    between, first, last




In [2]:
# Function to convert drug names to valid R variable names
make_valid_names <- function(names_vector) {
  # Replace invalid characters with underscores
  valid_names <- gsub("[^[:alnum:]_]", "_", names_vector)
  
  # Ensure the names start with a letter (required for R variable names)
  valid_names <- make.names(valid_names)
  
  return(valid_names)
}

In [3]:
# Function to clean the column names
clean_column_names <- function(names_vector) {
  # Remove "factor(valid_drug_names)X" prefix
  cleaned_names <- gsub("^factor\\(valid_drug_names\\)X", "", names_vector)
  
  # Replace multiple underscores with a single underscore
  cleaned_names <- gsub("_+", "_", cleaned_names)
  
  return(cleaned_names)
}

In [4]:
# Function to clean the column names
clean_contrast_names <- function(names_vector) {
  # Remove leading underscores (or other unwanted characters) and clean the names
  cleaned_names <- gsub("^_", "", names_vector)  # Remove leading underscores
  cleaned_names <- make.names(cleaned_names)  # Ensure valid R names
  
  # Optionally, remove additional unwanted characters if needed
  cleaned_names <- gsub("[^[:alnum:]_]", "_", cleaned_names)  # Replace non-alphanumeric characters with underscores
  
  return(cleaned_names)
}

In [5]:
# Set the path to your folder
folder_path <- "/lustre/groups/ml01/workspace/manuel.gander/data/prelimma/mD2"

# Get all files in the folder
files <- list.files(path = folder_path, full.names = TRUE)
sample_names <- sub(".*/(.*)_.*\\.csv", "\\1", files)
celllines = sort(unique(unname(sample_names)))
celllines

[1] "A172"      "A427"      "A498"      "A549"      "AN3CA"     "AsPC1"    
 [7] "BT474"     "C32"       "C33A"      "CFPAC1"    "CHP212"    "COLO205"  
[13] "H4"        "HCT15"     "HEC1A"     "HepG2C3A"  "HOP62"     "HS578T"   
[19] "Hs766T"    "HT29"      "hTERTHPNE" "J82"       "KATOIII"   "LoVo"     
[25] "LOXIMVI"   "LS180"     "MIAPaCa2"  "NCIH1573"  "NCIH1792"  "NCIH2030" 
[31] "NCIH2122"  "NCIH23"    "NCIH2347"  "NCIH460"   "NCIH596"   "NCIH661"  
[37] "Panc0327"  "PANC1"     "RKO"       "RPMI7951"  "SHP77"     "SKMEL2"   
[43] "SNU1"      "SNU423"    "SW1088"    "SW1271"    "SW1417"    "SW48"     
[49] "SW480"     "SW900"

In [7]:
for (cellline in celllines){
    tryCatch({
    file0 = '/lustre/groups/ml01/workspace/manuel.gander/data/prelimma/mD2/'
    path = paste0(file0, cellline)

    X <- as.matrix(fread(paste0(path, "_X.csv"), header = FALSE, skip = 1))
    var <- read_csv(paste0(path, "_var.csv"))
    obs <- read_csv(paste0(path, "_obs.csv"))

    valid_drug_names <- make_valid_names(obs$drugname_drugconc)
    plate <- factor(obs$plate)

    # Create the design matrix based on the cell types
    design_matrix <- model.matrix(~ 0 + factor(valid_drug_names) + plate)
    colnames(design_matrix) <- clean_column_names(colnames(design_matrix))
    colnames(design_matrix) <- clean_contrast_names(colnames(design_matrix))


    # Create DGEList object
    d0 <- DGEList(counts = t(X))
    keep_genes <- filterByExpr(d0, design = design_matrix)
    d0 <- d0[keep_genes, , keep.lib.sizes = FALSE]
    d0 <- calcNormFactors(d0)

    # Perform the Voom transformation and fit the model
    v <- voom(d0, design = design_matrix, plot = FALSE)
    fit <- lmFit(v, design_matrix)

    # Output the results
    summary(fit)

    # get proper DMSO name
    conditions <- setdiff(unique(colnames(design_matrix)), "DMSO_TF_0_0_uM_")

    # Initialize a list to store results
    de_results_list <- list()

    # Loop through conditions
    for (condition in conditions) {
        cat("Doing condition", condition, "\n")
        flush.console()  # Force the output to be printed immediately
        # Create contrast formula
        contrast_formula <- paste0(condition, " - ", "DMSO_TF_0_0_uM_")
        contr <- makeContrasts(contrasts = contrast_formula, levels = colnames(design_matrix))

        # Fit the contrast and apply eBayes
        contrast_fit <- contrasts.fit(fit, contr)
        contrast_fit <- eBayes(contrast_fit, robust = TRUE)

        # Get the top table of results
        top_table <- topTable(contrast_fit, number = Inf, sort.by = "none", adjust.method = "BH")
        rownames(top_table) <- var$gene_name[keep_genes]

        # Add additional metadata
        top_table$gene <- rownames(top_table)
        top_table$condition <- condition
        top_table$control_type <- "[('DMSO_TF', 0.0, 'uM')]"
        top_table$cellline <- cellline

        # Store in list (only if significant)
        filtered_table <- top_table[top_table$adj.P.Val < 0.05, ]
        if(nrow(filtered_table) > 0) {
            de_results_list[[condition]] <- filtered_table
        }
    }

    de_results_combined <- do.call(rbind, de_results_list)

    output_file <- paste0("/lustre/groups/ml01/workspace/manuel.gander/data/postlimma/mDf2/", 
        cellline, "_conditions.csv")
    write.csv(conditions, output_file, row.names = FALSE)

    output_file <- paste0("/lustre/groups/ml01/workspace/manuel.gander/data/postlimma/mDf2/", 
        cellline, "_differential_expression_results.csv")
    write.csv(de_results_combined, output_file, row.names = FALSE)
    
    })
}

Rows: 62710 Columns: 1
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr (1): gene_name

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 1339 Columns: 7
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr (5): BARCODE_SUB_LIB_ID, sample, cell_name, bc1_well, drugname_drugconc
dbl (2): n_cells, plate

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Doing condition R_Verapamil_hydrochloride_0_05_uM_ 
Doing condition R_Verapamil_hydrochloride_0_5_uM_ 
Doing condition R_Verapamil_hydrochloride_5_0_uM_ 
Doing condition S_Crizotinib_0_05_uM_ 
Doing condition S_Crizotinib_0_5_uM_ 
Doing condition S_Crizotinib_5_0_uM_ 
Doing condition X18β_Glycyrrhetinic_acid_0_05_uM_ 
Doing condition X18β_Glycyrrhetinic_acid_0_5_uM_ 
Doing condition X18β_Glycyrrhetinic_acid_5_0_uM_ 
Doing condition X4EGI_1_0_05_uM_ 
Doing condition X4EGI_1_0_5_uM_ 
Doing condition X4EGI_1_5_0_uM_ 
Doing condition X5_Azacytidine_0_05_uM_ 
Doing condition X5_Azacytidine_0_5_uM_ 
Doing condition X5_Azacytidine_5_0_uM_ 
Doing condition X5_Fluorouracil_0_05_uM_ 
Doing condition X5_Fluorouracil_0_5_uM_ 
Doing condition X5_Fluorouracil_5_0_uM_ 
Doing condition X8_Hydroxyquinoline_0_05_uM_ 
Doing condition X8_Hydroxyquinoline_0_5_uM_ 
Doing condition X8_Hydroxyquinoline_5_0_uM_ 
Doing condition X9_ING_41_0_05_uM_ 
Doing condition X9_ING_41_0_5_uM_ 
Doing condition X9_ING_41_5_

Doing condition Bosentan_hydrate_0_5_uM_ 
Doing condition Bosentan_hydrate_5_0_uM_ 
Doing condition Brimonidine_0_05_uM_ 
Doing condition Brimonidine_0_5_uM_ 
Doing condition Brimonidine_5_0_uM_ 
Doing condition Brivudine_0_05_uM_ 
Doing condition Brivudine_0_5_uM_ 
Doing condition Brivudine_5_0_uM_ 
Doing condition Budesonide_0_05_uM_ 
Doing condition Budesonide_0_5_uM_ 
Doing condition Budesonide_5_0_uM_ 
Doing condition Busulfan_0_05_uM_ 
Doing condition Busulfan_0_5_uM_ 
Doing condition Busulfan_5_0_uM_ 
Doing condition c_Kit_IN_1_0_05_uM_ 
Doing condition c_Kit_IN_1_0_5_uM_ 
Doing condition c_Kit_IN_1_5_0_uM_ 
Doing condition Cabozantinib_S_malate_0_05_uM_ 
Doing condition Cabozantinib_S_malate_0_5_uM_ 
Doing condition Cabozantinib_S_malate_5_0_uM_ 
Doing condition Canagliflozin_0_05_uM_ 
Doing condition Canagliflozin_0_5_uM_ 
Doing condition Canagliflozin_5_0_uM_ 
Doing condition Canagliflozin_hemihydrate_0_05_uM_ 
Doing condition Canagliflozin_hemihydrate_0_5_uM_ 
Doing conditio

Doing condition Entrectinib_0_05_uM_ 
Doing condition Entrectinib_0_5_uM_ 
Doing condition Entrectinib_5_0_uM_ 
Doing condition Epirubicin_hydrochloride_0_05_uM_ 
Doing condition Epirubicin_hydrochloride_0_5_uM_ 
Doing condition Epirubicin_hydrochloride_5_0_uM_ 
Doing condition Eplerenone_0_05_uM_ 
Doing condition Eplerenone_0_5_uM_ 
Doing condition Eplerenone_5_0_uM_ 
Doing condition Erdafitinib_0_05_uM_ 
Doing condition Erdafitinib_5_0_uM_ 
Doing condition ERK5_IN_2_0_05_uM_ 
Doing condition ERK5_IN_2_0_5_uM_ 
Doing condition ERK5_IN_2_5_0_uM_ 
Doing condition Erlotinib_0_05_uM_ 
Doing condition Erlotinib_0_5_uM_ 
Doing condition Erlotinib_5_0_uM_ 
Doing condition Erythromycin_0_05_uM_ 
Doing condition Erythromycin_0_5_uM_ 
Doing condition Erythromycin_5_0_uM_ 
Doing condition Esmolol_hydrochloride_0_05_uM_ 
Doing condition Esmolol_hydrochloride_0_5_uM_ 
Doing condition Esmolol_hydrochloride_5_0_uM_ 
Doing condition Estrone_sulfate_potassium_0_05_uM_ 
Doing condition Estrone_sulfate_

Doing condition Lenalidomide_hemihydrate_0_05_uM_ 
Doing condition Lenalidomide_hemihydrate_0_5_uM_ 
Doing condition Lenalidomide_hemihydrate_5_0_uM_ 
Doing condition Levobupivacaine_hydrochloride_0_05_uM_ 
Doing condition Levobupivacaine_hydrochloride_0_5_uM_ 
Doing condition Levobupivacaine_hydrochloride_5_0_uM_ 
Doing condition Lidocaine_hydrochloride_0_05_uM_ 
Doing condition Lidocaine_hydrochloride_0_5_uM_ 
Doing condition Lidocaine_hydrochloride_5_0_uM_ 
Doing condition Ligustrazine_0_05_uM_ 
Doing condition Ligustrazine_0_5_uM_ 
Doing condition Ligustrazine_5_0_uM_ 
Doing condition Lipoic_acid_0_05_uM_ 
Doing condition Lipoic_acid_0_5_uM_ 
Doing condition Lipoic_acid_5_0_uM_ 
Doing condition LJI308_0_05_uM_ 
Doing condition LJI308_0_5_uM_ 
Doing condition LJI308_5_0_uM_ 
Doing condition Lonafarnib_0_05_uM_ 
Doing condition Lonafarnib_0_5_uM_ 
Doing condition Lonafarnib_5_0_uM_ 
Doing condition Loperamide_hydrochloride_0_05_uM_ 
Doing condition Loperamide_hydrochloride_0_5_uM_ 
D

...

Doing condition PF_06260933_0_05_uM_ 
Doing condition PF_06260933_0_5_uM_ 
Doing condition PF_06260933_5_0_uM_ 
Doing condition PH_797804_0_05_uM_ 
Doing condition PH_797804_0_5_uM_ 
Doing condition PH_797804_5_0_uM_ 
Doing condition Phenylephrine_hydrochloride_0_05_uM_ 
Doing condition Phenylephrine_hydrochloride_0_5_uM_ 
Doing condition Phenylephrine_hydrochloride_5_0_uM_ 
Doing condition Phenytoin_sodium_0_05_uM_ 
Doing condition Phenytoin_sodium_0_5_uM_ 
Doing condition Phenytoin_sodium_5_0_uM_ 
Doing condition Pimitespib_0_05_uM_ 
Doing condition Pimitespib_0_5_uM_ 
Doing condition Pimitespib_5_0_uM_ 
Doing condition Pimozide_0_05_uM_ 
Doing condition Pimozide_0_5_uM_ 
Doing condition Pimozide_5_0_uM_ 
Doing condition Pioglitazone_0_05_uM_ 
Doing condition Pioglitazone_0_5_uM_ 
Doing condition Pioglitazone_5_0_uM_ 
Doing condition Piroxicam_0_05_uM_ 
Doing condition Piroxicam_0_5_uM_ 
Doing condition Piroxicam_5_0_uM_ 
Doing condition Pitavastatin_Calcium_0_05_uM_ 
Doing condition

Doing condition Thymopentin_5_0_uM_ 
Doing condition Tirabrutinib_0_05_uM_ 
Doing condition Tirabrutinib_0_5_uM_ 
Doing condition Tirabrutinib_5_0_uM_ 
Doing condition Tirabrutinib_hydrochloride_0_05_uM_ 
Doing condition Tirabrutinib_hydrochloride_0_5_uM_ 
Doing condition Tirabrutinib_hydrochloride_5_0_uM_ 
Doing condition Tofacitinib_0_05_uM_ 
Doing condition Tofacitinib_0_5_uM_ 
Doing condition Tofacitinib_5_0_uM_ 
Doing condition Tofacitinib_citrate_0_05_uM_ 
Doing condition Tofacitinib_citrate_0_5_uM_ 
Doing condition Tofacitinib_citrate_5_0_uM_ 
Doing condition Tolcapone_0_05_uM_ 
Doing condition Tolcapone_0_5_uM_ 
Doing condition Tolcapone_5_0_uM_ 
Doing condition Tolmetin_0_05_uM_ 
Doing condition Tolmetin_0_5_uM_ 
Doing condition Tolmetin_5_0_uM_ 
Doing condition Tomivosertib_0_05_uM_ 
Doing condition Tomivosertib_0_5_uM_ 
Doing condition Tomivosertib_5_0_uM_ 
Doing condition Topotecan_hydrochloride_0_05_uM_ 
Doing condition Topotecan_hydrochloride_0_5_uM_ 
Doing condition Topo

Rows: 62710 Columns: 1
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr (1): gene_name

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 1336 Columns: 7
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr (5): BARCODE_SUB_LIB_ID, sample, cell_name, bc1_well, drugname_drugconc
dbl (2): n_cells, plate

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Doing condition R_Verapamil_hydrochloride_0_05_uM_ 
Doing condition R_Verapamil_hydrochloride_0_5_uM_ 
Doing condition R_Verapamil_hydrochloride_5_0_uM_ 
Doing condition S_Crizotinib_0_05_uM_ 
Doing condition S_Crizotinib_0_5_uM_ 
Doing condition S_Crizotinib_5_0_uM_ 
Doing condition X18β_Glycyrrhetinic_acid_0_05_uM_ 
Doing condition X18β_Glycyrrhetinic_acid_0_5_uM_ 
Doing condition X18β_Glycyrrhetinic_acid_5_0_uM_ 
Doing condition X4EGI_1_0_05_uM_ 
Doing condition X4EGI_1_0_5_uM_ 
Doing condition X4EGI_1_5_0_uM_ 
Doing condition X5_Azacytidine_0_05_uM_ 
Doing condition X5_Azacytidine_0_5_uM_ 
Doing condition X5_Azacytidine_5_0_uM_ 


IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

